# Python Chapter 2: All Programs
This notebook contains all the examples from Python Phase 2, ready to be run in Google Colab.

## 09_Concurrency - 11e_threading_basic.py

In [ ]:
import threading, time

def handle_client(name, wait_time):
    print(f"Serving {name} (waiting {wait_time}s)...")
    time.sleep(wait_time) # Simulating I/O
    print(f"Finished {name}!")

requests = [("Client 1", 3), ("Client 2", 1), ("Client 3", 1)]
threads = []

for name, wait in requests:
    t = threading.Thread(target=handle_client, args=(name, wait))
    threads.append(t)
    t.start()

for t in threads: t.join()


## 09_Concurrency - 11e_threading_v2.py

In [ ]:
import threading
import time

# -------------------------------------------------------------------------
# Web Server Simulation: Multithreaded Client Request Handling
#
# Real-World Scenario:
# - Client 1 sends a request that triggers a slow database query (takes 3s).
# - Client 2 sends a fast request (takes 1s).
# - Client 3 sends a fast request (takes 1s).
#
# Without Threads (Sequential):
#   Client 2 is blocked and must wait 3s before its request even starts!
#   Total time: 3s + 1s + 1s = 5s.
#
# With Multithreading:
#   While Thread-1 is WAITING on Client 1's I/O, Thread-2 immediately
#   serves Client 2, and Thread-3 serves Client 3 concurrently!
#   Total time: ~3s (the duration of the longest wait).
# -------------------------------------------------------------------------

def handle_client(client_name, wait_time):
    """Simulates a worker thread handling an incoming client request."""
    thread_name = threading.current_thread().name
    print(f"[{thread_name}] Received request from {client_name}...")
    print(f"[{thread_name}] {client_name} is WAITING on I/O ({wait_time}s)...")
    
    # time.sleep() simulates waiting for network/database/external API (I/O Bound)
    # Python releases the GIL during I/O sleep so other threads run concurrently!
    time.sleep(wait_time)
    
    print(f"[{thread_name}] --> [COMPLETED] Finished serving {client_name} in {wait_time}s\n")

if __name__ == "__main__":
    print("=" * 65)
    print(" WEB SERVER: THREAD POOL HANDLING CONCURRENT CLIENT REQUESTS")
    print("=" * 65)
    
    requests = [
        ("Client 1 (Slow Database Query)", 3),
        ("Client 2 (Fast Cache Lookup)", 1),
        ("Client 3 (Fast Web Page)", 1),
    ]

    start_time = time.time()

    # Dispatch each incoming client request to a separate worker thread
    threads = []
    for i, (client_name, wait_time) in enumerate(requests, start=1):
        t = threading.Thread(
            target=handle_client,
            args=(client_name, wait_time),
            name=f"Thread-{i}"
        )
        threads.append(t)
        t.start()
        # Small stagger (0.05s) to simulate clients arriving one after another
        time.sleep(0.05)

    # Wait for all threads to complete their work
    for t in threads:
        t.join()

    total_time = round(time.time() - start_time, 2)
    print("=" * 65)
    print(f" All clients served in {total_time}s (instead of 5.0s sequentially)!")
    print(" Key Takeaway:")
    print(" While Thread-1 was paused waiting on Client 1's I/O, Thread-2 and")
    print(" Thread-3 simultaneously served Client 2 and Client 3 without blocking.")
    print("=" * 65)


## 09_Concurrency - 11f_multiprocessing.py

In [ ]:
import multiprocessing
import time

def heavy_computation(num):
    print(f"Computing {num}...")
    result = sum(i*i for i in range(num))
    print(f"Result for {num}: {result}")

if __name__ == "__main__":
    processes = []
    for i in [10000000, 20000000]:
        p = multiprocessing.Process(target=heavy_computation, args=(i,))
        processes.append(p)
        p.start()
        
    for p in processes:
        p.join()
        
    print("All computations finished.")


## 10_HTTP_Fundamentals - 16_http_requests.py

In [ ]:
import requests

# 1. Simple GET request
print("--- 1. Making a GET Request ---")
url = "https://api.open-meteo.com/v1/forecast?latitude=52.52&longitude=13.41&current_weather=true"
response = requests.get(url)

# Print Status Code (200 OK means Success!)
print(f"Status Code: {response.status_code}")

# Print Headers
print(f"Content-Type: {response.headers.get('Content-Type')}")

# 2. Parsing the JSON response
if response.status_code == 200:
    print("\n--- 2. Parsing Nested JSON ---")
    data = response.json()  # Converts the JSON string into a Python Dictionary
    # Let's extract the specific piece of data we care about
    temp = data["current_weather"]["temperature"]
    windspeed = data["current_weather"]["windspeed"]
    
    print(f"Current Temperature: {temp}°C")
    print(f"Current Windspeed: {windspeed} km/h")
else:
    print("Failed to fetch data")


## 10_HTTP_Fundamentals - 16b_post_requests.py

In [ ]:
import requests
import json

print("--------------------------------------------------")
print(" HTTP POST REQUEST (Sending Data)")
print("--------------------------------------------------")

url = "https://jsonplaceholder.typicode.com/posts"

# The data we want to send (e.g., creating a new blog post)
data_to_send = {
    "title": "Learning Python Requests",
    "body": "POST requests are used to send data to a server.",
    "userId": 1
}

print(f"Sending POST request to: {url}")
print(f"Data payload: {data_to_send}\n")

# Make the POST request. The 'json' parameter automatically converts 
# our Python dictionary into a JSON string and sets the correct headers.
response = requests.post(url, json=data_to_send)

# 201 Created is the standard status code when a POST request successfully creates a resource
print(f"Status Code: {response.status_code}")

if response.status_code == 201:
    print("Success! The server accepted our data and created the resource.")
    
    # The server usually responds with the created object (including its new ID)
    response_data = response.json()
    print("\nServer Response:")
    print(json.dumps(response_data, indent=2))
else:
    print(f"Failed to create resource. Server responded with: {response.text}")

print("--------------------------------------------------")


## 11_Robust_APIs - 17_robust_requests.py

In [ ]:
import requests
import time

url = "https://api.open-meteo.com/v1/forecast?latitude=26.57&longitude=74.01&current_weather=true"
max_retries = 3

print("--- Robust API Call with Retries and Exception Handling ---")

for attempt in range(max_retries):
    try:
        print(f"Attempt {attempt + 1}...")
        
        # Add timeout to prevent the program from hanging forever
        response = requests.get(url, timeout=5)
        
        # Raise an exception if the status code isn't 200 OK
        response.raise_for_status() 
        
        # If we got here, it succeeded!
        temp = response.json()["current_weather"]["temperature"]
        print(f"Success! The temperature is {temp}°C in Rajashthan")
        break  # Exit the loop on success
        
    except requests.exceptions.Timeout:
        print(f"Error: The request timed out. Retrying...")
        time.sleep(2)  # Wait before trying again
        
    except requests.exceptions.RequestException as e:
        # RequestException catches any requests-related error (Connection error, 404, etc.)
        print(f"Fatal API Error: {e}")
        break  # Exit loop immediately for non-recoverable errors


## 12_Calling_LLMs - 12_calling_llm.py

In [ ]:
# 12_calling_llm.py
# Make sure to run: pip install openai python-dotenv
import os
from dotenv import load_dotenv
from openai import OpenAI

# Loads environment variables from a .env file
load_dotenv()

# The client automatically looks for the OPENAI_API_KEY environment variable
client = OpenAI()

def ask_assistant(prompt: str):
    print("Calling OpenAI API...")
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        print("\n--- Response ---")
        print(response.choices[0].message.content)
        print("----------------\n")
    except Exception as e:
        print(f"API call failed: {e}")

if __name__ == "__main__":
    # We will configure the .env file with our API key before running this.
    user_prompt = "What are the top 3 new developments in AI this month?"
    ask_assistant(user_prompt)


## 12_Calling_LLMs - 13_calling_openrouter.py

In [ ]:
# 13_calling_openrouter.py
# Make sure to run: pip install openai python-dotenv
import os
from dotenv import load_dotenv
from openai import OpenAI

# Loads environment variables from a .env file
load_dotenv()

# OpenRouter uses the OpenAI SDK format, just with a different base_url
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENRouter_API_KEY")
)

def ask_openrouter(prompt: str):
    print("Calling OpenRouter API...")
    try:
        response = client.chat.completions.create(
            model="meta-llama/llama-3.1-8b-instruct",
            messages=[
                {"role": "system", "content": "You are a helpful assistant.You provide summarized short answers."},
                {"role": "user", "content": prompt}
            ]
        )
        print("\n--- Response ---")
        print(response.choices[0].message.content)
        print("----------------\n")
    except Exception as e:
        print(f"API call failed: {e}")

if __name__ == "__main__":
    # We will configure the .env file with our API key before running this.
    user_prompt = "What are the fundamental concepts of AI?"
    ask_openrouter(user_prompt)


## 12_Calling_LLMs - 14_calling_claude.py

In [ ]:
# 14_calling_claude.py
# Make sure to run: pip install anthropic python-dotenv
import os
from dotenv import load_dotenv
from anthropic import Anthropic

# Loads environment variables from a .env file
load_dotenv()

# Initialize the Anthropic client (automatically looks for ANTHROPIC_API_KEY)
client = Anthropic()

def ask_claude(prompt: str):
    print("Calling Claude API...")
    try:
        response = client.messages.create(
            model="claude-3-haiku-20240307",
            max_tokens=1000,
            system="You are a helpful assistant.",
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        print("\n--- Response ---")
        print(response.content[0].text)
        print("----------------\n")
    except Exception as e:
        print(f"API call failed: {e}")

if __name__ == "__main__":
    # We will configure the .env file with our API key before running this.
    user_prompt = "What are the top 3 new developments in AI this month?"
    ask_claude(user_prompt)


## 12_Calling_LLMs - 15_api_file_refactor.py

In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open("prompt.txt", "r") as f:
    prompt = f.read().strip()

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    reply = response.choices[0].message.content

    with open("response.txt", "w") as f:
        f.write(reply)

    print("Response saved to response.txt")
except Exception as e:
    print(f"API call failed: {e}")


## 13_Weather_Practice - 18_resilient_weather_logger.py

In [ ]:
"""
PRACTICE ACTIVITY: Resilient Weather Logger

Goal:
1. Make a request to the Open-Meteo API for your local city's coordinates.
2. Use a try/except block and a timeout.
3. If successful, parse the JSON and write the temperature to 'weather_log.txt'.
4. If it fails, catch the error and write the error message to 'weather_log.txt'.
"""

import requests
import time

url = "https://api.open-meteo.com/v1/forecast?latitude=40.71&longitude=-74.00&current_weather=true" # Defaulting to NYC

try:
    print("Fetching weather data...")
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    
    data = response.json()
    temp = data["current_weather"]["temperature"]
    
    # Write success to log file
    with open("weather_log.txt", "a") as f:
        log_entry = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Success: Temperature in NYC is {temp}°C\n"
        f.write(log_entry)
        
    print("Weather logged successfully! Check weather_log.txt")
    
except requests.exceptions.RequestException as e:
    # Write failure to log file
    with open("weather_log.txt", "a") as f:
        error_entry = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] Error: Failed to fetch weather. Details: {e}\n"
        f.write(error_entry)
        
    print("Failed to fetch weather. Check weather_log.txt for details.")
